# Modul B · Kapitel 2 · Bonus — Hybrid Search

## Challenge: Zwei Suchverfahren zu einem Ergebnis verschmelzen


**Lernziel:** Du kannst zwei Suchverfahren zu einer Rangfolge verschmelzen — über eine gewichtete
Summe normalisierter Scores und über Reciprocal Rank Fusion — und misst, was das bringt.

Dieses Notebook ist freiwillig. Es arbeitet am Retrieval-Schritt der RAG-Kette:

```
Dokumente ──► Chunks ──► Embeddings ──► Vector Database ──► Retrieval ──► Prompt ──► Antwort
                                                             └─ hier ─┘
```

Die Ausgangslage: Über die Wissensbasis des Security Operations Center laufen zwei Suchen.
**BM25** vergleicht Wörter und findet, was wörtlich dasteht — Kennungen, Produktnamen,
Konfigurationsschlüssel. **Semantic Search** vergleicht Vektoren und findet, was gemeint ist,
auch wenn die Frage andere Wörter benutzt. Keines der beiden Verfahren ist dem anderen überlegen,
und sie liegen an verschiedenen Stellen daneben. Genau das ist der Fall für **Hybrid Search**.

Gerechnet wird ohne Sprachmodell: Alle Ergebnisse in diesem Notebook sind deterministisch.

### So funktioniert dieses Notebook

| Symbol | Bedeutung |
|:--:|---|
| 📖 | Erklärung — lesen |
| ▶️ | Fertiger Code — einfach ausführen (`Shift` + `Enter`) |
| 🛠️ | **Challenge** — hier schreibst du selbst Code |
| ✅ | Selbsttest — sagt dir sofort, ob deine Lösung stimmt |
| 💡 | **Lösung** — zum Aufklappen, wenn du nicht weiterkommst |

**Wichtig:** Führe die Zellen **von oben nach unten** aus. Spätere Zellen brauchen die
Funktionen, die du vorher schreibst.

Es sind insgesamt **5 Challenges**.

---
## 0 · Setup

▶️ Führe die beiden nächsten Codezellen aus.

Die erste holt die Pakete. Ein Sprachmodell ruft dieses Notebook nicht auf — gerechnet wird in
Python, und die Embeddings kommen aus dem Cache. Die LLM-Naht steht deshalb nicht hier, sondern
an einer einzigen Stelle in `helfer.py`:

```python
# Ollama, lokal. Für einen Provider: base_url und api_key tauschen, Modellname anpassen.
BASIS_URL = "http://localhost:11434/v1"
API_KEY = "ollama"                # Ollama prüft den Key nicht
MODELL = "qwen3.5:0.8b"           # 1,0 GB, läuft auf jedem Laptop
EMBEDDING_MODELL = "nomic-embed-text"

client = OpenAI(base_url=BASIS_URL, api_key=API_KEY)
```

Die Namen daraus werden importiert, nicht neu gesetzt. Ein Modellwechsel greift so überall.

In [ ]:
# ▶️ Pakete und Diagrammstil
import json
import re
import sys
from pathlib import Path

try:
    import chromadb
    from rank_bm25 import BM25Okapi
except ImportError:
    %pip install -q chromadb rank-bm25
    import chromadb
    from rank_bm25 import BM25Okapi

import matplotlib.pyplot as plt

# Einheitliche Farben für alle Diagramme in diesem Notebook
BLAU, ORANGE, TEAL, GRAU = "#2563eb", "#e8590c", "#0d9488", "#6b7280"

plt.rcParams.update({
    "figure.figsize": (8, 4.5),
    "figure.dpi": 110,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "axes.edgecolor": GRAU,
    "axes.grid": True,
    "axes.axisbelow": True,
    "grid.color": "#e5e7eb",
    "axes.titlesize": 13,
    "axes.titleweight": "bold",
    "font.size": 11,
})

print("Setup fertig ✔")

▶️ Die zweite Zelle lädt `helfer.py`, holt die 90 vorbereiteten Chunks und legt die
Vector Database an. Von dort kommt auch der Name des Embedding-Modells. Die Embeddings selbst
liegen in `daten/embedding_cache.json`, deshalb dauert das nur einen Moment.

In [ ]:
# ▶️ helfer.py finden, Chunks laden, Vector Database bauen
for kandidat in [Path.cwd(), *Path.cwd().parents]:
    if (kandidat / "helfer.py").exists():
        sys.path.insert(0, str(kandidat))
        break

import helfer
from helfer import EMBEDDING_MODELL

chunks = helfer.lade_chunks()
fragen = helfer.lade_fragen()
sammlung = helfer.baue_chroma(chunks, neu=True)

CHUNK_NACH_ID = {c["chunk_id"]: c for c in chunks}

print(f"Embedding-Modell: {EMBEDDING_MODELL}")
print(f"{len(chunks)} Chunks aus {len({c['dok_id'] for c in chunks})} Dokumenten")
print(f"Collection {sammlung.name!r}: {sammlung.count()} Einträge unter {helfer.CHROMA_PFAD}")
print(f"{len(fragen)} Evaluationsfragen")

---
## 1 · Zwei Suchen nebeneinander

📖 Beide Verfahren sind fertig. Sie bekommen hier eine gemeinsame Form, damit sie sich
verschmelzen lassen:

* `bm25_scores(frage)` und `semantische_scores(frage)` geben je ein Dict `{chunk_id: score}`
  über **alle 90 Chunks** zurück — nicht nur über die besten fünf.
* `rangfolge(scores)` macht daraus eine Liste von `chunk_id`, bester Treffer zuerst.
* `beste_n(scores, n)` gibt die besten `n` Chunks als Trefferliste zurück, jeder Chunk um sein
  Feld `score` ergänzt.

Beide Score-Tabellen werden zwischengespeichert. Weiter unten wird jede Frage mehrere hundert Mal
gesucht; ohne den Zwischenspeicher würde jedes Mal derselbe Index abgefragt.

▶️ Die nächste Zelle stellt beide Suchen bereit.

In [ ]:
# ▶️ BM25 über die Chunk-Texte, Semantic Search über die Collection
STOPPWOERTER = {
    "der", "die", "das", "des", "dem", "den", "ein", "eine", "einer", "eines", "einem", "einen",
    "und", "oder", "aber", "auch", "als", "wie", "was", "wer", "wo", "wann", "warum", "welche",
    "ist", "sind", "war", "waren", "wird", "werden", "wurde", "wurden", "hat", "haben", "kann",
    "in", "im", "an", "am", "auf", "aus", "bei", "mit", "nach", "von", "vom", "vor", "zu", "zum",
    "zur", "für", "über", "unter", "durch", "gegen", "ohne", "um", "bis", "seit", "je", "pro",
    "ich", "du", "er", "sie", "es", "wir", "ihr", "man", "sich", "dieser", "diese", "dieses",
    "nicht", "kein", "keine", "nur", "noch", "schon", "so", "dass", "wenn", "dann", "sein",
}


def tokenisiere(text):
    """Zerlegt einen Text in kleingeschriebene Suchwörter ohne Stoppwörter."""
    woerter = re.findall(r"[a-z0-9äöüß]+(?:-[a-z0-9äöüß]+)*", text.lower())
    return [w for w in woerter if w not in STOPPWOERTER]


bm25 = BM25Okapi([tokenisiere(c["text"]) for c in chunks], k1=1.5, b=0.75)

_bm25_speicher, _semantik_speicher = {}, {}


def bm25_scores(frage):
    """BM25-Score jedes Chunks zu einer Frage, als Dict {chunk_id: score}."""
    if frage not in _bm25_speicher:
        werte = bm25.get_scores(tokenisiere(frage))
        _bm25_speicher[frage] = {c["chunk_id"]: float(w) for c, w in zip(chunks, werte)}
    return dict(_bm25_speicher[frage])


def semantische_scores(frage):
    """Kosinus-Ähnlichkeit jedes Chunks zu einer Frage, als Dict {chunk_id: score}."""
    if frage not in _semantik_speicher:
        roh = sammlung.query(query_embeddings=helfer.embed([frage]), n_results=len(chunks))
        _semantik_speicher[frage] = {kennung: 1 - abstand for kennung, abstand
                                     in zip(roh["ids"][0], roh["distances"][0])}
    return dict(_semantik_speicher[frage])


def rangfolge(scores):
    """Die chunk_id aller Chunks, nach Score absteigend. Gleichstand: nach chunk_id."""
    return sorted(scores, key=lambda kennung: (-scores[kennung], kennung))


def beste_n(scores, n=5):
    """Die n besten Chunks als Trefferliste, jeder Chunk um sein Feld score ergänzt."""
    return [{**CHUNK_NACH_ID[kennung], "score": scores[kennung]}
            for kennung in rangfolge(scores)[:n]]


def suche_bm25(frage, n=5):
    """Die n Chunks mit dem höchsten BM25-Score."""
    return beste_n(bm25_scores(frage), n)


def suche_semantisch(frage, n=5):
    """Die n Chunks mit der höchsten Kosinus-Ähnlichkeit."""
    return beste_n(semantische_scores(frage), n)


print(f"BM25-Index über {len(chunks)} Chunks, Collection mit {sammlung.count()} Einträgen")

📖 Zwei Fragen zeigen die Ausgangslage. Beide haben genau ein erwartetes Dokument, und bei beiden
findet nur eines der Verfahren es auf Platz 1.

**Frage K** enthält eine exakte Kennung: `CVE-2026-4410`. Vier CVE-Advisories in der Wissensbasis
sind sich als Text sehr ähnlich — dieselbe Gliederung, dieselben Fachbegriffe. Was sie
unterscheidet, ist eine Nummer.

**Frage S** ist umformuliert: Sie fragt nach der Gefährlichkeit einer Lücke im VPN-Zugang. In den
Dokumenten heißt das *CVSS*, *Authentication Bypass* und *NorthPeak SecureGate*.

▶️ Beide Rangfolgen, für beide Fragen.

In [ ]:
# ▶️ Dieselbe Frage, zwei Verfahren
FRAGE_K = "Welcher Workaround schützt vor CVE-2026-4410, solange kein Patch verfügbar ist?"
FRAGE_S = "Wie gefährlich ist die Lücke im VPN-Zugang von NorthPeak?"
ERWARTET = {FRAGE_K: ["cve-2026-4410"], FRAGE_S: ["cve-2026-3224"]}


def zeige_rangfolge(name, treffer, erwartet):
    """Fünf Treffer mit Rang, Score und einer Marke für das erwartete Dokument."""
    print(f"  — {name} —")
    for rang, t in enumerate(treffer, start=1):
        marke = "✔" if t["dok_id"] in erwartet else " "
        print(f"   {marke} {rang}. {t['score']:7.3f}  {t['chunk_id']:<32} {t['dok_id']}")


for frage in (FRAGE_K, FRAGE_S):
    print(f"❓ {frage}")
    print(f"   erwartet: {', '.join(ERWARTET[frage])}")
    zeige_rangfolge("BM25", suche_bm25(frage, n=5), ERWARTET[frage])
    zeige_rangfolge("Semantic Search", suche_semantisch(frage, n=5), ERWARTET[frage])
    print()

📖 **Frage K.** BM25 setzt die beiden Chunks aus `cve-2026-4410` auf Platz 1 und 2, mit deutlichem
Abstand: Die Kennung steht nur in diesem Advisory, ihre Inverse Document Frequency ist
entsprechend hoch. Semantic Search liefert an Platz 1 und 2 Chunks aus zwei **anderen**
Advisories. Die drei besten Scores liegen keine drei Tausendstel auseinander — für das
Embedding-Modell sind `CVE-2026-4410` und `CVE-2026-3224` fast derselbe Text.

**Frage S.** Umgekehrt. Semantic Search setzt das erwartete Advisory auf Platz 1, 2 und 3. BM25
setzt einen Post-Mortem-Chunk nach oben, in dem *VPN* und *NorthPeak* wörtlich vorkommen; das
Advisory folgt erst auf Platz 3.

Keines der beiden Verfahren ist also das bessere. Sie scheitern an verschiedenen Stellen — und
das ist die Voraussetzung dafür, dass eine Verschmelzung überhaupt etwas bringen kann. Zwei
Verfahren, die dieselben Fehler machen, ergeben zusammen dieselben Fehler.

---
## 2 · Warum die Scores nicht direkt vergleichbar sind

📖 Der naheliegende Gedanke ist, die beiden Scores zu addieren. Das geht nicht, weil sie auf
verschiedenen Skalen liegen.

| | BM25 | Kosinus-Ähnlichkeit |
|---|---|---|
| Wertebereich | 0 bis offen nach oben | −1 bis 1, in der Praxis eng um einen Mittelwert |
| Bedeutung von 0 | kein gemeinsames Suchwort | rechtwinklig, also unähnlich |
| hängt ab von | Termfrequenz, IDF, Chunklänge | Richtung zweier Vektoren |

▶️ Beide Fragen, beide Skalen nebeneinander.

In [ ]:
# ▶️ Die Wertebereiche beider Verfahren
for frage in (FRAGE_K, FRAGE_S):
    keyword, semantik = bm25_scores(frage), semantische_scores(frage)
    null = sum(1 for w in keyword.values() if w == 0)
    print(f"❓ {frage[:66]}")
    print(f"   BM25       {min(keyword.values()):7.3f} bis {max(keyword.values()):7.3f}"
          f"   ·   Spannweite {max(keyword.values()) - min(keyword.values()):6.3f}"
          f"   ·   {null} von {len(keyword)} Chunks bei genau 0")
    print(f"   Kosinus    {min(semantik.values()):7.3f} bis {max(semantik.values()):7.3f}"
          f"   ·   Spannweite {max(semantik.values()) - min(semantik.values()):6.3f}")
    print()

📖 Die BM25-Spannweite ist bei der ersten Frage rund fünfzigmal, bei der zweiten rund
zwanzigmal so groß wie die der Kosinus-Ähnlichkeit. Eine Summe aus beiden Rohwerten wäre eine
BM25-Rangfolge mit einer sehr kleinen Störung.

Auffällig ist die dritte Zahl: Für die meisten Chunks ist der BM25-Score **genau 0**. Sie teilen
kein einziges Suchwort mit der Frage. Die Kosinus-Ähnlichkeit hat diesen Fall nicht — jeder Chunk
bekommt einen Wert, auch wenn er nichts mit der Frage zu tun hat.

Der übliche Weg heißt **Min-Max-Normalisierung**. Sie bildet den kleinsten Wert einer Liste auf 0
ab, den größten auf 1, alles dazwischen linear:

```
normalisiert(s) = (s − min) / (max − min)
```

Danach lassen sich beide Seiten gewichtet addieren:

```
hybrid(chunk) = α · BM25norm(chunk)  +  (1 − α) · Kosinusnorm(chunk)
```

`α` ist der einzige neue Parameter. `α = 1` ist reines BM25, `α = 0` ist reine Semantic Search,
dazwischen wird gemischt.

### 🛠️ Challenge 1: Normalisieren und gewichtet addieren

Zwei Funktionen.

**`normalisiere(scores)`** bekommt eine Liste von Zahlen und gibt eine Liste gleicher Länge
zurück, alle Werte zwischen 0 und 1. Zwei Randfälle:

* Eine leere Liste ergibt eine leere Liste.
* Sind **alle Werte gleich**, ist `max − min` gleich 0. Dann darf nicht geteilt werden; gib für
  jeden Wert `0.0` zurück. Kein Wert sticht heraus, also hebt sich keiner ab.

**`verschmelze_scores(keyword, semantisch, alpha=0.5)`** bekommt zwei Dicts `{chunk_id: score}`,
normalisiert **jedes für sich** und gibt ein Dict über die **Vereinigung** beider Schlüssel
zurück. Ein Chunk, der nur in einer der beiden Tabellen steht, wird auf der anderen Seite mit
`0.0` gewertet.

*Tipp: `dict(zip(keyword, normalisiere(list(keyword.values()))))` normalisiert eine ganze Tabelle
auf einmal — `zip` läuft über die Schlüssel, und die Reihenfolge von `keys()` und `values()` ist
in Python dieselbe. Mit `tabelle.get(kennung, 0.0)` fängst du fehlende Chunks ab.*

In [ ]:
def normalisiere(scores):
    """Bildet eine Liste von Scores linear auf 0…1 ab."""
    # TODO 1: leere Liste → leere Liste
    # TODO 2: min und max bestimmen; sind sie gleich, für jeden Wert 0.0 zurückgeben
    # TODO 3: sonst (s - min) / (max - min) für jeden Wert
    raise NotImplementedError("Challenge 1: normalisiere() implementieren")


def verschmelze_scores(keyword, semantisch, alpha=0.5):
    """Gewichtete Summe zweier getrennt normalisierter Score-Tabellen."""
    # TODO 4: beide Tabellen einzeln normalisieren
    # TODO 5: über die Vereinigung beider Schlüssel
    #         alpha * keyword + (1 - alpha) * semantisch rechnen
    raise NotImplementedError("Challenge 1: verschmelze_scores() implementieren")


In [ ]:
# ✅ Selbsttest
assert normalisiere([1.0, 3.0, 2.0]) == [0.0, 1.0, 0.5], "Kleinster Wert 0, größter 1"
assert normalisiere([-4.0, 0.0, 4.0]) == [0.0, 0.5, 1.0], "Auch mit negativen Werten"
assert normalisiere([]) == [], "Leere Liste bleibt leer"
assert normalisiere([5.0]) == [0.0], "Ein einziger Wert: min und max fallen zusammen"
assert normalisiere([2.0, 2.0, 2.0]) == [0.0, 0.0, 0.0], "Alle gleich: keine Division durch null"

# zwei kleine Trefferlisten von Hand
K = {"a": 10.0, "b": 5.0, "c": 0.0}          # normalisiert: a 1,0   b 0,5   c 0,0
S = {"a": 0.25, "b": 0.75, "d": 0.50}        # normalisiert: a 0,0   b 1,0   d 0,5

nur_keyword = verschmelze_scores(K, S, alpha=1.0)
assert nur_keyword["a"] == 1.0 and nur_keyword["c"] == 0.0, "alpha = 1: nur BM25 zählt"
assert nur_keyword["d"] == 0.0, "d steht nicht in der Keyword-Tabelle"

nur_semantik = verschmelze_scores(K, S, alpha=0.0)
assert nur_semantik["b"] == 1.0, "alpha = 0: nur die semantische Suche zählt"
assert nur_semantik["a"] == 0.0, "a ist dort der schwächste der drei Treffer"

mitte = verschmelze_scores(K, S, alpha=0.5)
assert set(mitte) == {"a", "b", "c", "d"}, "Die Vereinigung beider Trefferlisten"
assert mitte == {"a": 0.5, "b": 0.75, "c": 0.0, "d": 0.25}, "Von Hand nachgerechnet"
assert max(mitte, key=mitte.get) == "b", "b steht in beiden Listen weit oben und gewinnt"

print("✅ Challenge 1 gelöst")
zahl = lambda tabelle, kennung: f"{tabelle[kennung]:.2f}" if kennung in tabelle else "—"

print(f"{'Chunk':<8}{'BM25':>8}{'Kosinus':>10}{'α = 0':>8}{'α = 0,5':>10}{'α = 1':>8}")
print("-" * 52)
for kennung in ["a", "b", "c", "d"]:
    print(f"{kennung:<8}{zahl(K, kennung):>8}{zahl(S, kennung):>10}"
          f"{nur_semantik[kennung]:>8.2f}{mitte[kennung]:>10.2f}{nur_keyword[kennung]:>8.2f}")

<details>
<summary>💡 Lösung aufklappen — erst selbst probieren!</summary>

```python
def normalisiere(scores):
    """Bildet eine Liste von Scores linear auf 0…1 ab."""
    if not scores:
        return []
    tief, hoch = min(scores), max(scores)
    if hoch == tief:
        return [0.0 for _ in scores]
    return [(s - tief) / (hoch - tief) for s in scores]


def verschmelze_scores(keyword, semantisch, alpha=0.5):
    """Gewichtete Summe zweier getrennt normalisierter Score-Tabellen."""
    a = dict(zip(keyword, normalisiere(list(keyword.values()))))
    b = dict(zip(semantisch, normalisiere(list(semantisch.values()))))
    kennungen = list(keyword) + [k for k in semantisch if k not in keyword]
    return {k: alpha * a.get(k, 0.0) + (1 - alpha) * b.get(k, 0.0) for k in kennungen}
```

Der Sonderfall `hoch == tief` ist nicht nur eine Absicherung gegen die Division durch null. Er
tritt real auf: Enthält kein einziger Chunk ein Suchwort der Frage, sind alle BM25-Scores 0. Die
Rückgabe `0.0` sorgt dafür, dass diese Seite der Summe dann nichts beiträgt.

Die Normalisierung hängt an der Menge, über die sie läuft. Hier ist das der ganze Bestand aus 90
Chunks. Läuft sie nur über die besten fünf, bekommt der beste Treffer immer 1,0 — auch wenn er
schlecht ist.

</details>

▶️ Dieselben beiden Fragen, jetzt mit der gewichteten Summe.

In [ ]:
# ▶️ Die verschmolzene Rangfolge, α = 0,5
for frage in (FRAGE_K, FRAGE_S):
    verschmolzen = verschmelze_scores(bm25_scores(frage), semantische_scores(frage), alpha=0.5)
    print(f"❓ {frage}")
    print(f"   erwartet: {', '.join(ERWARTET[frage])}")
    zeige_rangfolge("gewichtete Summe, α = 0,5", beste_n(verschmolzen, 5), ERWARTET[frage])
    print()

---
## 3 · Reciprocal Rank Fusion

📖 Die gewichtete Summe braucht vergleichbare Scores, und die entstehen erst durch die
Normalisierung. Reciprocal Rank Fusion umgeht das: Sie wirft die Scores weg und rechnet nur mit
**Rängen**.

```
                      1
RRF(d) =   Σ    ─────────────
         Listen   k + rang(d)
```

Für jedes Dokument `d` wird über alle Rangfolgen summiert. `rang(d)` ist die Position in der
jeweiligen Liste, beginnend bei 1. Steht `d` in einer Liste nicht, trägt diese Liste nichts bei.

Ein Rang ist eine dimensionslose Zahl. Damit ist es gleichgültig, ob eine Suche Scores zwischen 0
und 10 liefert oder zwischen 0,56 und 0,77 — und ob sie überhaupt Scores liefert.

**Was `k` tut.** Der Summand für Platz 1 ist `1/(k+1)`, für Platz 2 `1/(k+2)`. Der Abstand
zwischen zwei benachbarten Plätzen hängt also von `k` ab:

| `k` | Platz 1 | Platz 2 | Abstand | Wirkung |
|---:|---:|---:|---:|---|
| 0 | 1,000 | 0,500 | 0,500 | ein erster Platz wiegt genauso viel wie zwei zweite |
| 10 | 0,0909 | 0,0833 | 0,0076 | die vorderen Plätze zählen deutlich mehr |
| 60 | 0,0164 | 0,0161 | 0,0003 | die vorderen Plätze zählen ähnlich viel |

Je größer `k`, desto flacher die Kurve und desto mehr zählt, dass ein Dokument **in beiden**
Listen vorne steht, statt in einer ganz vorne. `k = 60` ist der Wert aus der Arbeit, die das
Verfahren eingeführt hat, und der Standard in fast jeder Implementierung.

### 🛠️ Challenge 2: Reciprocal Rank Fusion

Schreibe `rrf(rangfolgen, k=60)`. Eingabe ist eine **Liste von Rangfolgen**; jede Rangfolge ist
eine Liste von `chunk_id`, bester Treffer zuerst. Rückgabe ist ein Dict `{chunk_id: punkte}` mit
der Summe aus der Formel oben.

Ein Chunk, der in mehreren Rangfolgen vorkommt, sammelt aus jeder einen Summanden. Ein Chunk, der
nur in einer vorkommt, bekommt nur diesen einen.

*Tipp: `enumerate(rangfolge, start=1)` liefert Rang und `chunk_id` in einem. Mit
`punkte.get(kennung, 0.0)` fängst du den ersten Eintrag ab, ohne vorher zu prüfen.*

In [ ]:
def rrf(rangfolgen, k=60):
    """Reciprocal Rank Fusion: Summe von 1 / (k + Rang) über alle Rangfolgen."""
    # TODO 1: ein leeres Dict für die Punkte anlegen
    # TODO 2: über jede Rangfolge und darin über Rang und chunk_id laufen
    # TODO 3: 1 / (k + rang) auf den bisherigen Punktestand des Chunks addieren
    raise NotImplementedError("Challenge 2: rrf() implementieren")


In [ ]:
# ✅ Selbsttest
A = ["chunk-a", "chunk-c"]      # Verfahren 1
B = ["chunk-b", "chunk-c"]      # Verfahren 2

# von Hand mit k = 1:
#   chunk-a  nur in A, Platz 1      →  1/(1+1)              = 0,5000
#   chunk-b  nur in B, Platz 1      →  1/(1+1)              = 0,5000
#   chunk-c  in beiden auf Platz 2  →  1/(1+2) + 1/(1+2)    = 0,6667
punkte = rrf([A, B], k=1)
assert set(punkte) == {"chunk-a", "chunk-b", "chunk-c"}, "Die Vereinigung beider Rangfolgen"
assert punkte["chunk-a"] == 0.5, "Platz 1 in einer Liste: 1/(1+1)"
assert punkte["chunk-b"] == 0.5, "Platz 1 in der anderen Liste: 1/(1+1)"
assert abs(punkte["chunk-c"] - 2 / 3) < 1e-12, "Zweimal Platz 2: 1/3 + 1/3"
assert max(punkte, key=punkte.get) == "chunk-c", "Zweimal Platz 2 schlägt einmal Platz 1"

# der Standardwert k = 60
standard = rrf([["x", "y", "z"]])
assert abs(standard["x"] - 1 / 61) < 1e-12, "Platz 1 ergibt 1/(60+1)"
assert abs(standard["y"] - 1 / 62) < 1e-12, "Platz 2 ergibt 1/(60+2)"
assert standard["x"] > standard["y"] > standard["z"], "Eine einzelne Rangfolge bleibt erhalten"

assert rrf([]) == {}, "Keine Rangfolge, keine Punkte"
assert rrf([[], []]) == {}, "Leere Rangfolgen, keine Punkte"

# k steuert, wie viel ein erster Platz gegenüber Übereinstimmung wiegt
ohne_k, mit_k = rrf([A, B], k=0), rrf([A, B], k=60)
assert ohne_k["chunk-a"] == ohne_k["chunk-c"], \
    "Bei k = 0 ist ein erster Platz genau zwei zweite Plätze wert"
assert mit_k["chunk-c"] > mit_k["chunk-a"], \
    "Bei k = 60 gewinnt, wer in beiden Rangfolgen vorne steht"

print("✅ Challenge 2 gelöst")
print(f"{'Chunk':<10}{'in A':>7}{'in B':>7}{'k = 0':>10}{'k = 1':>10}{'k = 60':>10}")
print("-" * 54)
for kennung in ["chunk-a", "chunk-b", "chunk-c"]:
    platz = lambda liste: str(liste.index(kennung) + 1) if kennung in liste else "—"
    print(f"{kennung:<10}{platz(A):>7}{platz(B):>7}"
          f"{ohne_k[kennung]:>10.4f}{punkte[kennung]:>10.4f}{mit_k[kennung]:>10.4f}")

<details>
<summary>💡 Lösung aufklappen — erst selbst probieren!</summary>

```python
def rrf(rangfolgen, k=60):
    """Reciprocal Rank Fusion: Summe von 1 / (k + Rang) über alle Rangfolgen."""
    punkte = {}
    for liste in rangfolgen:
        for rang, kennung in enumerate(liste, start=1):
            punkte[kennung] = punkte.get(kennung, 0.0) + 1 / (k + rang)
    return punkte
```

Die Funktion nimmt eine Liste von Rangfolgen, nicht genau zwei. Damit lassen sich auch drei oder
vier Verfahren verschmelzen — der Code ändert sich dafür nicht.

Ein `collections.defaultdict(float)` spart das `get()`. Das Ergebnis ist dasselbe.

</details>

▶️ Was `k` an einer echten Frage bewirkt.

In [ ]:
# ▶️ Dieselben zwei Rangfolgen, vier Werte für k
keyword = rangfolge(bm25_scores(FRAGE_K))
semantik = rangfolge(semantische_scores(FRAGE_K))

print(f"❓ {FRAGE_K}")
print(f"   erwartet: {', '.join(ERWARTET[FRAGE_K])}")
print()
print(f"{'Chunk':<24}{'BM25':>7}{'Kosinus':>9}" + "".join(f"{'k = ' + str(k):>12}" for k in (0, 10, 60, 200)))
print("-" * 100)

kandidaten = list(dict.fromkeys(keyword[:4] + semantik[:4]))
punkte_je_k = {k: rrf([keyword, semantik], k=k) for k in (0, 10, 60, 200)}

for kennung in kandidaten:
    zeile = f"{kennung:<24}{keyword.index(kennung) + 1:>7}{semantik.index(kennung) + 1:>9}"
    for k in (0, 10, 60, 200):
        werte = punkte_je_k[k]
        platz = rangfolge(werte).index(kennung) + 1
        zeile += f"{platz}. ({werte[kennung]:.4f})".rjust(12)
    marke = "  ← erwartet" if CHUNK_NACH_ID[kennung]["dok_id"] in ERWARTET[FRAGE_K] else ""
    print(zeile + marke)

📖 Die beiden linken Spalten sind die Ränge, aus denen gerechnet wird. `cve-2026-3224#05` steht
bei BM25 auf Platz 3 und bei Semantic Search auf Platz 1, `cve-2026-4410#02` umgekehrt auf Platz
1 und Platz 4.

An den ersten drei Plätzen ändert `k` hier nichts — über sie sind sich die beiden Verfahren zu
einig. Sichtbar wird die Wirkung weiter hinten, an `cve-2025-9042#04`: Platz 2 bei Semantic
Search, Platz 16 bei BM25. Bei `k = 0` reicht sein zweiter Platz, um vor `cve-2026-1187#04` zu
kommen, das in beiden Listen mittelmäßig liegt. Bei `k = 60` fällt `cve-2025-9042#04` dahinter
zurück, bei `k = 200` auf Platz 7 — je größer `k`, desto stärker zählt auch ein schlechter Rang
mit.

Die absoluten Zahlen werden dabei klein und bedeutungslos: Ein RRF-Wert von 0,0323 sagt für sich
genommen nichts. Nur die Reihenfolge zählt.

Das ist zugleich die Kehrseite: RRF kennt keine Abstände. Ein Treffer, der mit großem Vorsprung
der beste ist, und einer, der knapp vorne liegt, sind beide „Platz 1".

---
## 4 · Die Hybrid-Suche

📖 Beide Verschmelzungen kommen jetzt in **eine** Suchfunktion. Sie führt beide Suchen aus,
verschmilzt sie nach dem gewählten Verfahren und gibt die besten `n` Chunks zurück.

### 🛠️ Challenge 3: `hybrid_suche()`

Schreibe `hybrid_suche(frage, n=5, verfahren="rrf", alpha=0.5)`:

1. Beide Score-Tabellen holen: `bm25_scores(frage)` und `semantische_scores(frage)`.
2. Bei `verfahren="rrf"`: aus beiden Tabellen mit `rangfolge()` je eine Rangfolge machen und
   `rrf()` darauf anwenden.
3. Bei `verfahren="summe"`: `verschmelze_scores()` mit `alpha` anwenden.
4. Bei jedem anderen Wert einen `ValueError` werfen — ein Tippfehler im Verfahrensnamen soll
   nicht stillschweigend zu einem falschen Ergebnis führen.
5. Die besten `n` mit `beste_n()` zurückgeben.

*Tipp: `beste_n(punkte, n)` erledigt Sortierung und Trefferformat. `raise ValueError(f"…")` wirft
den Fehler.*

In [ ]:
def hybrid_suche(frage, n=5, verfahren="rrf", alpha=0.5):
    """Verschmilzt BM25 und Semantic Search und gibt die n besten Chunks zurück."""
    # TODO 1: beide Score-Tabellen holen
    # TODO 2: verfahren == "rrf"   → rrf() auf beide Rangfolgen anwenden
    # TODO 3: verfahren == "summe" → verschmelze_scores() mit alpha anwenden
    # TODO 4: sonst einen ValueError werfen
    # TODO 5: die besten n mit beste_n() zurückgeben
    raise NotImplementedError("Challenge 3: hybrid_suche() implementieren")


In [ ]:
# ✅ Selbsttest
treffer = hybrid_suche(FRAGE_K, n=5, verfahren="rrf")

assert len(treffer) == 5, f"Fünf Treffer erwartet, {len(treffer)} bekommen"
assert all({"chunk_id", "dok_id", "titel", "text", "score"} <= set(t) for t in treffer), \
    "Jeder Treffer ist der Chunk plus das Feld score"
assert treffer == sorted(treffer, key=lambda t: -t["score"]), "Absteigend sortiert"
assert len({t["chunk_id"] for t in treffer}) == 5, "Kein Chunk darf doppelt vorkommen"
assert len(hybrid_suche(FRAGE_K, n=3)) == 3, "n steuert die Trefferzahl"

# die Randfälle von alpha müssen die Einzelverfahren genau reproduzieren
kennungen = lambda liste: [t["chunk_id"] for t in liste]
assert kennungen(hybrid_suche(FRAGE_K, n=5, verfahren="summe", alpha=1.0)) \
    == kennungen(suche_bm25(FRAGE_K, n=5)), "α = 1 muss die BM25-Rangfolge ergeben"
assert kennungen(hybrid_suche(FRAGE_S, n=5, verfahren="summe", alpha=0.0)) \
    == kennungen(suche_semantisch(FRAGE_S, n=5)), "α = 0 muss die semantische Rangfolge ergeben"

try:
    hybrid_suche(FRAGE_K, verfahren="magie")
except ValueError:
    pass
else:
    raise AssertionError("Ein unbekanntes Verfahren muss einen ValueError werfen")

assert treffer[0]["dok_id"] in ERWARTET[FRAGE_K] or treffer[1]["dok_id"] in ERWARTET[FRAGE_K], \
    "Das erwartete Advisory gehört unter die ersten beiden Treffer"

print("✅ Challenge 3 gelöst")
helfer.zeige_treffer(treffer)

<details>
<summary>💡 Lösung aufklappen — erst selbst probieren!</summary>

```python
def hybrid_suche(frage, n=5, verfahren="rrf", alpha=0.5):
    """Verschmilzt BM25 und Semantic Search und gibt die n besten Chunks zurück."""
    keyword = bm25_scores(frage)
    semantisch = semantische_scores(frage)

    if verfahren == "rrf":
        punkte = rrf([rangfolge(keyword), rangfolge(semantisch)])
    elif verfahren == "summe":
        punkte = verschmelze_scores(keyword, semantisch, alpha=alpha)
    else:
        raise ValueError(f"Unbekanntes Verfahren {verfahren!r} — erlaubt sind 'rrf' und 'summe'")

    return beste_n(punkte, n)
```

`alpha` wird bei `verfahren="rrf"` nicht benutzt. Das ist Absicht: Die Signatur bleibt für beide
Verfahren dieselbe, sodass die Messung weiter unten beide gleich behandeln kann.

Dass `α = 1` genau die BM25-Rangfolge ergibt, ist kein Zufall. Die Min-Max-Normalisierung ist
streng monoton — sie verschiebt und staucht die Werte, dreht aber keine Reihenfolge um.

</details>

▶️ Die beiden Fragen aus Abschnitt 1, jetzt durch alle vier Verfahren.

In [ ]:
# ▶️ Vier Verfahren, zwei Fragen: auf welchem Platz steht das erwartete Dokument?
VERFAHREN = [
    ("BM25", lambda f, n: suche_bm25(f, n=n)),
    ("Semantic Search", lambda f, n: suche_semantisch(f, n=n)),
    ("Summe α = 0,5", lambda f, n: hybrid_suche(f, n=n, verfahren="summe", alpha=0.5)),
    ("RRF k = 60", lambda f, n: hybrid_suche(f, n=n, verfahren="rrf")),
]

for frage in (FRAGE_K, FRAGE_S):
    print(f"❓ {frage}")
    print(f"   erwartet: {', '.join(ERWARTET[frage])}")
    for name, suche in VERFAHREN:
        treffer = suche(frage, 5)
        plaetze = [str(r) for r, t in enumerate(treffer, start=1)
                   if t["dok_id"] in ERWARTET[frage]]
        print(f"   {name:<18} Platz 1: {treffer[0]['chunk_id']:<32}"
              f"erwartetes Dokument auf Platz {', '.join(plaetze) or '— (nicht unter den ersten fünf)'}")
    print()

📖 Beide Verschmelzungen setzen bei beiden Fragen einen Chunk aus dem erwarteten Dokument unter
die ersten beiden Plätze. Kein Einzelverfahren schafft das für beide Fragen.

Das sind zwei Fragen. Ob daraus ein Ergebnis wird, entscheidet die Messung.

---
## 5 · Messen über zwei Fragensätze

📖 Gemessen wird mit **Recall@k**: dem Anteil der Fragen, bei denen unter den ersten `k` Treffern
mindestens ein Chunk aus einem erwarteten Dokument liegt.

Ein Fragensatz allein reicht dafür nicht. `daten/fragen.json` enthält zehn Fragen, die die Wörter
der Dokumente benutzen — *Firewall-Logs*, *Notfall-Patch*, *SEV-1*, `CVE-2026-3224`. Das ist der
Fall, für den eine Suche über Wortübereinstimmung gebaut ist. Ein zweiter Satz stellt sechs
derselben Sachverhalte in anderen Worten. Erst beide zusammen zeigen, ob eine Verschmelzung
etwas bringt oder nur den Fragensatz bedient, auf dem sie gemessen wird.

▶️ Beide Fragensätze und die Messfunktion.

In [ ]:
# ▶️ Der zweite Fragensatz und Recall@k
UMFORMULIERT = [
    ("Wie gefährlich ist die Lücke im VPN-Zugang von NorthPeak?", ["cve-2026-3224"]),
    ("Wie viel Zeit bleibt für ein dringendes Sicherheitsupdate?", ["runbook-patch-management"]),
    ("Wer stuft einen Vorfall als besonders schwerwiegend ein?", ["runbook-incident-response"]),
    ("Wie lange bleiben die Aufzeichnungen der Firewall gespeichert?", ["policy-logging-und-aufbewahrung"]),
    ("Was geschieht mit dem Zugang, wenn jemand die Firma verlässt?", ["runbook-zugriffskontrolle"]),
    ("Wie viel Last hält die SIEM-Plattform im Alltag aus?", ["systemdoku-sentinelgrid"]),
]

UMFORMULIERTE_FRAGEN = [{"frage": f, "erwartete_dok_ids": e} for f, e in UMFORMULIERT]
ALLE_FRAGEN = fragen + UMFORMULIERTE_FRAGEN

FRAGENSAETZE = [
    ("daten/fragen.json", fragen),
    ("dieselben Sachverhalte, umformuliert", UMFORMULIERTE_FRAGEN),
    ("beide Sätze zusammen", ALLE_FRAGEN),
]


def recall_at_k(satz, suchfunktion, k=5):
    """Anteil der Fragen, bei denen unter den ersten k Treffern ein erwartetes Dokument steht."""
    getroffen = 0
    for frage in satz:
        gefunden = {t["dok_id"] for t in suchfunktion(frage["frage"], k)}
        if gefunden & set(frage["erwartete_dok_ids"]):
            getroffen += 1
    return getroffen / len(satz)


for name, satz in FRAGENSAETZE:
    print(f"{name:<40}{len(satz):>3} Fragen")

In [ ]:
# ▶️ Vier Verfahren, zwei Fragensätze, drei Werte für k
for name, satz in FRAGENSAETZE:
    print(f"{name}  ({len(satz)} Fragen)")
    print(f"  {'Verfahren':<20}{'Recall@1':>10}{'Recall@3':>10}{'Recall@5':>10}")
    print("  " + "-" * 50)
    for verfahren, suche in VERFAHREN:
        werte = [recall_at_k(satz, suche, k) for k in (1, 3, 5)]
        print(f"  {verfahren:<20}" + "".join(f"{w:>10.0%}" for w in werte))
    print()

In [ ]:
# ▶️ Dieselben Zahlen als Diagramm
fig, achsen = plt.subplots(1, 2, figsize=(11, 4.5), sharey=True)
namen = [v for v, _ in VERFAHREN]

for achse, (titel, satz) in zip(achsen, FRAGENSAETZE[:2]):
    for i, (k, farbe) in enumerate(zip((1, 3, 5), (BLAU, ORANGE, TEAL))):
        hoehen = [recall_at_k(satz, suche, k) for _, suche in VERFAHREN]
        achse.bar([p + (i - 1) * 0.27 for p in range(len(namen))], hoehen, 0.25,
                  label=f"Recall@{k}", color=farbe)
    achse.set_xticks(range(len(namen)))
    achse.set_xticklabels(namen, rotation=18, ha="right")
    achse.set_title(f"{titel}  ({len(satz)} Fragen)")
    achse.set_ylim(0, 1.30)

achsen[0].set_ylabel("Recall")
achsen[0].legend(loc="upper left", ncol=3, frameon=False)
plt.tight_layout()
plt.show()

📖 Die Zahlen ordnen sich so ein.

**Bei den Fragen aus `fragen.json` gewinnt Hybrid nichts.** BM25 liegt dort bei Recall@1 vorn,
die gewichtete Summe zieht gleich, RRF fällt um eine Frage zurück. Recall@5 ist bei allen vier
Verfahren bei 100 Prozent — mehr als „alles gefunden" geht nicht. Diese Fragen benutzen die
Wörter der Dokumente; hier ist nichts zu reparieren.

**Bei den umformulierten Fragen gewinnt Hybrid auf Platz 1.** BM25 trifft dort eine von sechs,
Semantic Search zwei, RRF drei. Das ist der Fall, für den Fusion gebaut ist: Ein Dokument, das in
beiden Rangfolgen im Mittelfeld steht, rückt vor eines, das nur in einer ganz oben steht.

**Unter den ersten fünf dreht sich das Bild.** Dort liegt BM25 mit 83 Prozent vor Semantic Search
mit 67 Prozent, und RRF landet ebenfalls bei 83 Prozent — die Verschmelzung übernimmt den
besseren der beiden Werte, ohne ihn zu übertreffen. Die gewichtete Summe verhält sich genauso.

**Bei Recall@3 fällt RRF zurück**, über beide Sätze zusammen hinter beide Einzelverfahren. Zwei
Fragen, deren erwartetes Dokument bei BM25 auf Platz 3 steht, rutschen durch die Verschmelzung
auf Platz 5. Die gewichtete Summe hat diesen Einbruch nicht.

**Über beide Sätze zusammen** stehen die beiden Verschmelzungen bei Recall@1 vor beiden
Einzelverfahren. Das ist das Ergebnis, das zählt: nicht besser auf einem Fragensatz, sondern
weniger schlecht über beide.

Ein Vorbehalt gehört dazu. Es sind 16 Fragen. Eine einzelne Frage, die den Platz wechselt,
verschiebt den Recall um sechs Prozentpunkte. Die Unterschiede zwischen den vier Verfahren liegen
teilweise in dieser Größenordnung — sie zeigen eine Richtung, keine belastbare Rangfolge.

Noch eine Eigenheit der Daten: BM25 vergibt an die meisten Chunks den Score 0, weil sie kein
Suchwort der Frage enthalten. In der Rangfolge stehen diese Chunks trotzdem auf irgendeinem
Platz — hier nach `chunk_id` sortiert. RRF rechnet mit diesen Plätzen, als wären sie eine
Aussage. Wer das vermeiden will, verschmilzt nur die besten 20 oder 50 Treffer jedes Verfahrens.

---
## 6 · Wie viel Gewicht bekommt welches Verfahren?

📖 `α = 0,5` war eine Setzung, keine Messung. Die gewichtete Summe hat den Parameter, also lässt
er sich durchfahren: von `α = 0` — reine Semantic Search — bis `α = 1` — reines BM25.

### 🛠️ Challenge 4: Die α-Kurve

Schreibe `alpha_kurve(satz, k=1, alphas=ALPHAS)`. Die Funktion misst für jeden Wert aus `alphas`
den Recall@k über den Fragensatz und gibt eine **Liste von Paaren** `(alpha, recall)` zurück, in
der Reihenfolge von `alphas`.

Gesucht wird mit `hybrid_suche(..., verfahren="summe", alpha=alpha)`, gemessen mit
`recall_at_k()`. Die Suchfunktion, die `recall_at_k` erwartet, nimmt zwei Argumente: `frage`
und `k`.

*Tipp: Ein `lambda` in einer Schleife merkt sich die Variable, nicht ihren Wert. Binde `alpha`
deshalb als Standardargument: `lambda f, n, a=alpha: hybrid_suche(f, n=n, verfahren="summe",
alpha=a)`.*

In [ ]:
ALPHAS = [i / 20 for i in range(21)]      # 0,00  0,05  0,10  …  1,00


def alpha_kurve(satz, k=1, alphas=ALPHAS):
    """Recall@k über einen Fragensatz für jeden Wert von alpha."""
    # TODO 1: über alle Werte in alphas laufen
    # TODO 2: je Wert eine Suchfunktion bauen, die alpha an hybrid_suche durchreicht
    # TODO 3: recall_at_k damit messen und (alpha, recall) sammeln
    raise NotImplementedError("Challenge 4: alpha_kurve() implementieren")


In [ ]:
# ✅ Selbsttest
kurve = alpha_kurve(ALLE_FRAGEN, k=1)

assert len(kurve) == len(ALPHAS), f"{len(ALPHAS)} Messpunkte erwartet, {len(kurve)} bekommen"
assert [a for a, _ in kurve] == ALPHAS, "Die alpha-Werte in der Reihenfolge von ALPHAS"
assert all(0.0 <= r <= 1.0 for _, r in kurve), "Recall liegt zwischen 0 und 1"

# die Enden der Kurve müssen genau die Einzelverfahren treffen
werte = dict(kurve)
assert werte[1.0] == recall_at_k(ALLE_FRAGEN, lambda f, n: suche_bm25(f, n=n), k=1), \
    "Bei α = 1 muss dasselbe herauskommen wie bei reinem BM25"
assert werte[0.0] == recall_at_k(ALLE_FRAGEN, lambda f, n: suche_semantisch(f, n=n), k=1), \
    "Bei α = 0 muss dasselbe herauskommen wie bei reiner Semantic Search"
assert max(r for _, r in kurve) >= werte[0.0], "Irgendwo dazwischen darf es nicht schlechter sein"

print("✅ Challenge 4 gelöst")
bestes = max(kurve, key=lambda paar: (paar[1], -paar[0]))
print(f"Recall@1 über {len(ALLE_FRAGEN)} Fragen:")
print(f"  α = 0,00 (nur Semantic)  {werte[0.0]:.0%}")
print(f"  α = 1,00 (nur BM25)      {werte[1.0]:.0%}")
print(f"  bestes α = {bestes[0]:.2f}          {bestes[1]:.0%}")

<details>
<summary>💡 Lösung aufklappen — erst selbst probieren!</summary>

```python
ALPHAS = [i / 20 for i in range(21)]      # 0,00  0,05  0,10  …  1,00


def alpha_kurve(satz, k=1, alphas=ALPHAS):
    """Recall@k über einen Fragensatz für jeden Wert von alpha."""
    kurve = []
    for alpha in alphas:
        def suche(frage, n, a=alpha):
            return hybrid_suche(frage, n=n, verfahren="summe", alpha=a)
        kurve.append((alpha, recall_at_k(satz, suche, k=k)))
    return kurve
```

`ALPHAS` wird aus ganzen Zahlen gebildet, nicht mit einer Fließkomma-Schleife. `0.05` lässt sich
binär nicht exakt darstellen; zwanzigmal addiert ergibt es nicht 1,0. `i / 20` trifft die
Randwerte genau — und darauf verlässt sich der Selbsttest.

</details>

▶️ Die Kurve für Recall@1, @3 und @5, gemessen über beide Fragensätze zusammen.

In [ ]:
# ▶️ Recall über alpha
plt.figure(figsize=(9, 4.8))
optima = {}

for k, farbe in zip((1, 3, 5), (BLAU, ORANGE, TEAL)):
    kurve = alpha_kurve(ALLE_FRAGEN, k=k)
    plt.plot([a for a, _ in kurve], [r for _, r in kurve],
             marker="o", markersize=4, color=farbe, label=f"Recall@{k}")
    optima[k] = max(kurve, key=lambda paar: (paar[1], -paar[0]))

plt.axvline(optima[1][0], color=GRAU, linestyle=":")
plt.text(optima[1][0] + 0.015, 0.44, f"bestes α für Recall@1: {optima[1][0]:.2f}".replace(".", ","), color=GRAU)
plt.xlabel("α — Gewicht der Keyword-Suche  (0 = nur Semantic, 1 = nur BM25)")
plt.ylabel("Recall")
plt.ylim(0.20, 1.05)
plt.title(f"Gewichtete Summe über {len(ALLE_FRAGEN)} Fragen")
plt.legend(loc="lower center", ncol=3, frameon=False)
plt.show()

print(f"{'k':>3}{'bestes α':>11}{'Recall dort':>14}{'α = 0':>9}{'α = 0,5':>10}{'α = 1':>9}")
print("-" * 56)
for k in (1, 3, 5):
    werte = dict(alpha_kurve(ALLE_FRAGEN, k=k))
    alpha_stern, bester = optima[k]
    print(f"{k:>3}{alpha_stern:>11.2f}{bester:>14.0%}"
          f"{werte[0.0]:>9.0%}{werte[0.5]:>10.0%}{werte[1.0]:>9.0%}")

📖 Drei Dinge stehen in dieser Kurve.

**Das Optimum liegt nicht am Rand.** Für Recall@1 ist der beste gemessene Wert `α = 0,10`, also
ein deutliches Übergewicht der semantischen Seite bei spürbarem Beitrag der Wortsuche. Jeder
Wert zwischen 0,05 und 0,85 ist besser als beide Ränder — das ist die Aussage des Abschnitts.

**Die Kurve ist nicht glatt, sie springt.** Zwischen `α = 0,10` und `α = 0,15` fällt Recall@1 um
sechs Prozentpunkte und steigt bei `α = 0,25` wieder. Sechs Prozentpunkte sind bei 16 Fragen
**genau eine Frage**. Was hier wie ein Optimum aussieht, ist die Stelle, an der eine einzelne
Frage ihren Platz wechselt.

**Recall@3 und Recall@5 sind fast flach.** Sie kennen nur zwei Niveaus und einen Sprung
dazwischen. Je größer `k`, desto weniger hängt vom Gewicht ab: Es geht dann nicht mehr um die
Reihenfolge, sondern nur noch darum, ob ein Chunk überhaupt in die Liste kommt.

Die praktische Folgerung ist unbequem. Ein `α`, das auf 16 Fragen ausgewählt wurde, ist nicht
optimiert, sondern an 16 Fragen angepasst. Für einen belastbaren Wert braucht es einen
Fragensatz, der groß genug ist, um einzelne Fragen unwichtig zu machen, und eine Trennung in
einen Satz zum Einstellen und einen zweiten zum Prüfen. Solange das fehlt, ist `α = 0,5` eine
ehrlichere Angabe als ein auf die zweite Nachkommastelle gemessener Wert.

---
## 7 · Wenn die Verschmelzung schadet

📖 Fusion kann ein Ergebnis auch verschlechtern. Der Mechanismus ist derselbe, der sie sonst
stark macht: Sie mittelt. Ein Verfahren, das einen Chunk sicher auf Platz 1 setzt, wird vom
anderen überstimmt, das ihn für mittelmäßig hält.

Um das zu sehen, braucht es den Platz des erwarteten Dokuments in der **vollständigen**
Rangfolge, nicht nur unter den ersten fünf.

### 🛠️ Challenge 5: Den Platz des erwarteten Dokuments finden

Schreibe `rang_des_erwarteten(treffer, erwartete_dok_ids)`. Die Funktion durchläuft die
Trefferliste und gibt die Position des **ersten** Treffers zurück, dessen `dok_id` unter den
erwarteten steht — gezählt ab 1. Kommt kein erwartetes Dokument vor, ist die Rückgabe `None`.

*Tipp: `enumerate(treffer, start=1)` liefert Rang und Treffer in einem. `return` beendet die
Schleife beim ersten Fund.*

In [ ]:
def rang_des_erwarteten(treffer, erwartete_dok_ids):
    """Platz des ersten Treffers aus einem erwarteten Dokument, sonst None."""
    # TODO 1: über die Treffer laufen, den Rang ab 1 mitzählen
    # TODO 2: beim ersten Treffer aus einem erwarteten Dokument den Rang zurückgeben
    # TODO 3: kommt keiner vor, None zurückgeben
    raise NotImplementedError("Challenge 5: rang_des_erwarteten() implementieren")


In [ ]:
# ✅ Selbsttest
PROBE = [{"dok_id": "policy-logging-und-aufbewahrung"},
         {"dok_id": "cve-2026-3224"},
         {"dok_id": "runbook-patch-management"},
         {"dok_id": "cve-2026-3224"}]

assert rang_des_erwarteten(PROBE, ["policy-logging-und-aufbewahrung"]) == 1, "Der erste Treffer"
assert rang_des_erwarteten(PROBE, ["runbook-patch-management"]) == 3, "Der dritte Treffer"
assert rang_des_erwarteten(PROBE, ["cve-2026-3224"]) == 2, "Das erste Vorkommen zählt, nicht das letzte"
assert rang_des_erwarteten(PROBE, ["cve-2026-3224", "runbook-patch-management"]) == 2, \
    "Mehrere erwartete Dokumente: das früheste"
assert rang_des_erwarteten(PROBE, ["gibt-es-nicht"]) is None, "Kein Vorkommen ergibt None"
assert rang_des_erwarteten([], ["cve-2026-3224"]) is None, "Leere Trefferliste ergibt None"

echt = rang_des_erwarteten(suche_bm25(FRAGE_K, n=len(chunks)), ERWARTET[FRAGE_K])
assert echt == 1, f"BM25 setzt das Advisory zu {FRAGE_K[:30]}… auf Platz 1, gemessen: {echt}"

print("✅ Challenge 5 gelöst")
for name, suche in VERFAHREN:
    platz = rang_des_erwarteten(suche(FRAGE_S, len(chunks)), ERWARTET[FRAGE_S])
    print(f"  {name:<18} Platz {platz}")

<details>
<summary>💡 Lösung aufklappen — erst selbst probieren!</summary>

```python
def rang_des_erwarteten(treffer, erwartete_dok_ids):
    """Platz des ersten Treffers aus einem erwarteten Dokument, sonst None."""
    for rang, t in enumerate(treffer, start=1):
        if t["dok_id"] in erwartete_dok_ids:
            return rang
    return None
```

`None` ist hier besser als `0` oder `-1`: Ein Rang 0 gibt es nicht, und `-1` würde sich in einer
Sortierung wie der beste Platz verhalten.

</details>

▶️ Alle 16 Fragen, vier Verfahren, jeweils der Platz des erwarteten Dokuments in der
vollständigen Rangfolge über alle 90 Chunks.

In [ ]:
# ▶️ Platz des erwarteten Dokuments, Frage für Frage
raenge = {}
for frage in ALLE_FRAGEN:
    raenge[frage["frage"]] = [rang_des_erwarteten(suche(frage["frage"], len(chunks)),
                                                  frage["erwartete_dok_ids"])
                              for _, suche in VERFAHREN]

kopf = f"{'Frage':<56}" + "".join(f"{n.split(' ')[0]:>10}" for n, _ in VERFAHREN)
print(kopf)
print("-" * len(kopf))

schlechter = []
for frage in ALLE_FRAGEN:
    werte = raenge[frage["frage"]]
    bestes_einzel = min(werte[0], werte[1])
    marke = ""
    if min(werte[2], werte[3]) > bestes_einzel:
        marke = "   ← Fusion schlechter als das bessere Einzelverfahren"
        schlechter.append(frage)
    print(f"{frage['frage'][:54]:<56}" + "".join(f"{w:>10}" for w in werte) + marke)

print()
print(f"{len(schlechter)} von {len(ALLE_FRAGEN)} Fragen werden durch die Verschmelzung schlechter.")

In [ ]:
# ▶️ Ein Fall im Detail
FEHLGRIFF = "Welchen CVSS-Wert hat CVE-2026-3224?"
erwartet = ["cve-2026-3224"]

print(f"❓ {FEHLGRIFF}")
print(f"   erwartet: {', '.join(erwartet)}")
print()
for name, suche in VERFAHREN:
    zeige_rangfolge(name, suche(FEHLGRIFF, 4), erwartet)
    print()

keyword, semantik = bm25_scores(FEHLGRIFF), semantische_scores(FEHLGRIFF)
ziel = "cve-2026-3224#01"
print(f"Der Chunk mit der Antwort ist {ziel}:")
print(f"   BM25             Platz {rangfolge(keyword).index(ziel) + 1:>2}   Score {keyword[ziel]:6.3f}")
print(f"   Semantic Search  Platz {rangfolge(semantik).index(ziel) + 1:>2}   Score {semantik[ziel]:6.3f}")

📖 Bei dieser Frage steht Semantic Search allein am besten da: `cve-2026-3224#01` — der Chunk mit
der Antwort — liegt auf Platz 1. BM25 setzt zwei andere davor: einen Abschnitt aus einem fremden
Advisory, in dem *CVSS-Wert* wörtlich steht, und den Kopf des Post-Mortems, das die Kennung
`CVE-2026-3224` wörtlich nennt.

Der Chunk mit der Antwort teilt mit der Frage kein einziges Suchwort. Er schreibt
`CVSS v3.1 Base Score | 9.8 (critical)`, und die Kennung steht im Titel des Dokuments, nicht im
Chunk-Text. Sein BM25-Score ist exakt 0.

Damit rechnet die gewichtete Summe: 0 auf der einen Seite, der Höchstwert auf der anderen, ergibt
0,50. Der BM25-Sieger liegt bei Semantic Search auf Platz 4 und kommt damit auf 0,93. Ein Chunk,
den beide Verfahren für ordentlich halten, schlägt einen, den nur eines für den besten hält.
Genau das ist der Preis der Mittelung: **Ein Verfahren, das in einem Einzelfall sicher richtig
liegt, kann sich gegen ein Verfahren, das durchgängig mittelmäßig liegt, nicht durchsetzen.**

Das ist kein Fehler in der Implementierung, sondern die Eigenschaft, die Fusion im Mittel besser
macht. Wer sie nicht will, braucht eine Fallunterscheidung vor der Suche: Enthält die Frage eine
exakte Kennung, gewinnt die Wortsuche; sonst wird verschmolzen. Diese Fallunterscheidung heißt
Query Routing und ist ein eigenes Verfahren mit eigenen Fehlerquellen.

▶️ Die letzte Zelle legt die Ergebnisse in `daten/bonus_hybrid_treffer.json` ab.

In [ ]:
# ▶️ Ergebnisse sichern
def als_eintrag(frage, erwartete_dok_ids):
    """Frage, erwartete Dokumente und die fünf besten Treffer je Verfahren."""
    return {
        "frage": frage,
        "erwartete_dok_ids": erwartete_dok_ids,
        "treffer": {
            name: [{"chunk_id": t["chunk_id"], "score": round(t["score"], 6)}
                   for t in suche(frage, 5)]
            for name, suche in VERFAHREN
        },
    }


ergebnis = {
    "verfahren": ["bm25-okapi", "cosine", "gewichtete-summe", "rrf"],
    "parameter": {"k1": 1.5, "b": 0.75, "alpha": 0.5, "rrf_k": 60,
                  "embedding_modell": EMBEDDING_MODELL},
    "recall": {
        name: {verfahren: {f"@{k}": round(recall_at_k(satz, suche, k), 4) for k in (1, 3, 5)}
               for verfahren, suche in VERFAHREN}
        for name, satz in FRAGENSAETZE
    },
    "alpha_kurve": {f"@{k}": [[a, round(r, 4)] for a, r in alpha_kurve(ALLE_FRAGEN, k=k)]
                    for k in (1, 3, 5)},
    "fragen": [als_eintrag(f["frage"], f["erwartete_dok_ids"]) for f in fragen],
    "umformuliert": [als_eintrag(f, e) for f, e in UMFORMULIERT],
}

ziel = helfer.DATEN / "bonus_hybrid_treffer.json"
ziel.write_text(json.dumps(ergebnis, ensure_ascii=False, indent=1) + "\n", encoding="utf-8")

print(f"{ziel.name}: {len(ergebnis['fragen'])} Fragen, "
      f"{len(ergebnis['umformuliert'])} umformulierte Fragen, "
      f"{len(VERFAHREN)} Verfahren")

---
## 8 · Was du gebaut hast

* `normalisiere()` — Min-Max auf 0…1, mit dem Sonderfall, dass alle Werte gleich sind.
* `verschmelze_scores()` — die gewichtete Summe zweier getrennt normalisierter Score-Tabellen,
  gesteuert über `α`.
* `rrf()` — Reciprocal Rank Fusion über beliebig viele Rangfolgen, ohne Normalisierung.
* `hybrid_suche()` — beide Suchen ausführen, verschmelzen, die besten `n` zurückgeben.
* `alpha_kurve()` und `rang_des_erwarteten()` — die Messung über zwei Fragensätze und der Blick
  auf die Fälle, in denen die Verschmelzung schadet.

Die Zahlen zum Mitnehmen: Über beide Fragensätze zusammen liegen beide Verschmelzungen bei
Recall@1 vor beiden Einzelverfahren. Bei den umformulierten Fragen ist RRF auf Platz 1 das beste
Verfahren, bei den Fragen aus `daten/fragen.json` gewinnt Hybrid nichts. Recall@5 bleibt über
alle vier Verfahren fast gleich — Fusion sortiert besser, sie findet nicht mehr.

### Was Hybrid Search kostet

| | einzelnes Verfahren | Hybrid |
|---|---|---|
| Indizes | einer | zwei, beide zu pflegen |
| Anfragen je Frage | eine | zwei, dazu die Verschmelzung |
| Parameter | `k1`, `b` **oder** das Embedding-Modell | beides, dazu `α` oder `k` |
| Latency | eine Suche | die langsamere der beiden |

Beide Indizes müssen denselben Stand haben. Ein Chunk, der nur in einem von beiden liegt, ist für
die Verschmelzung ein Chunk, den das andere Verfahren „nicht gefunden" hat — und wird
entsprechend abgewertet.

**Wann sich das lohnt:** wenn die Fragen aus beiden Welten kommen. Ein Bestand mit exakten
Kennungen, Versionsnummern und Fehlercodes, in dem zugleich frei formuliert gefragt wird, ist der
Fall dafür. Ein Bestand ohne Kennungen, in dem nur natürlichsprachlich gefragt wird, ist es
nicht — dort trägt die Wortsuche wenig bei und kostet einen zweiten Index.

Selbst bauen muss man das nicht: Vector Databases wie Weaviate, Qdrant oder Elasticsearch bringen
die Verschmelzung als Teil einer einzigen Anfrage mit, meist mit RRF als Voreinstellung und `α`
als Parameter — die Rechnung dahinter ist die aus diesem Notebook.

---
### 🔬 Bonus — ohne Lösung

**1. Re-Ranking mit einem Cross-Encoder.** Die verschmolzene Liste ist eine Vorauswahl. Ein
Cross-Encoder liest Frage und Chunk **gemeinsam** und gibt eine Relevanzzahl zurück — genauer als
jede Rangfolge aus getrennten Vektoren, aber zu langsam für 90 Chunks je Anfrage. Deshalb steht
er hinter der Verschmelzung. Vorgehen:

1. `hybrid_suche(frage, n=20)` als Kandidatenliste.
2. Die 20 Paare aus Frage und Chunk-Text durch ein Cross-Encoder-Modell schicken, etwa
   `cross-encoder/ms-marco-MiniLM-L-6-v2` über `sentence-transformers`. Ohne zusätzliches Paket
   geht es auch mit `helfer.frage_llm()`: Frage und Chunk in einen Prompt, Antwortformat eine
   Zahl von 0 bis 10.
3. Nach dieser Zahl neu sortieren und Recall@1 über beide Fragensätze messen.

Zwei Fragen dazu: Wie viel bringt das Re-Ranking gegenüber der verschmolzenen Liste — und wie
verhält sich der Gewinn zur zusätzlichen Latency, wenn 20 statt einem Modellaufruf nötig sind?

**2. Nur die besten Treffer verschmelzen.** In diesem Notebook läuft die Fusion über alle 90
Chunks. Im Betrieb liefert jede Suche nur ihre besten `m` Treffer, und verschmolzen wird die
Vereinigung. Vorgehen:

1. `hybrid_suche` um einen Parameter `kandidaten=20` erweitern: beide Score-Tabellen vor der
   Verschmelzung auf ihre besten `m` Einträge kürzen.
2. Recall@1, @3 und @5 für `m` in 10, 20, 30, 50 und 90 messen, für beide Verschmelzungen.
3. Prüfen, was mit einem Chunk passiert, der nur in einer der beiden Listen steht.

Interessant ist dabei, ab welchem `m` sich nichts mehr ändert — und ob ein kleines `m` das
Ergebnis verbessert, weil die willkürlich sortierten Chunks mit BM25-Score 0 gar nicht erst in
die Rechnung kommen.